# ⚡ Alur Kerja Agen Konkuren dengan Microsoft Foundry (Python)

## 📋 Tutorial Pemrosesan Paralel Lanjutan

Notebook ini menunjukkan **pola alur kerja konkuren** menggunakan Microsoft Agent Framework. Anda akan mempelajari cara membangun alur kerja pemrosesan paralel berperforma tinggi di mana beberapa agen AI dijalankan secara simultan, secara dramatis meningkatkan throughput dan memungkinkan proses bisnis multi-thread yang canggih.

> **Catatan migrasi:** Contoh ini sebelumnya merujuk ke Models GitHub. Models GitHub telah dihentikan (akan pensiun Juli 2026), jadi sekarang menggunakan **Microsoft Foundry** melalui `FoundryChatClient`, yang menargetkan Azure OpenAI **Responses API**.

## 🎯 Tujuan Pembelajaran

### 🚀 **Dasar Pemrosesan Konkuren**
- **Eksekusi Agen Paralel**: Jalankan beberapa agen secara simultan untuk efisiensi maksimum
- **Orkestrasi Alur Kerja**: Koordinasi operasi konkuren sambil menjaga konsistensi data
- **Optimasi Performa**: Capai percepatan signifikan melalui pemrosesan paralel
- **Manajemen Sumber Daya**: Gunakan sumber daya model AI secara efisien di operasi konkuren

### 🏗️ **Pola Konkuren Lanjutan**
- **Pemrosesan Fork-Join**: Membagi pekerjaan ke beberapa agen dan menggabungkan hasilnya
- **Paralelisme Pipa**: Tahapan eksekusi yang saling tumpang tindih untuk throughput berkelanjutan
- **Penyeimbangan Beban**: Distribusikan pekerjaan secara merata di antara sumber daya agen yang tersedia
- **Titik Sinkronisasi**: Koordinasi agen konkuren pada tahap kritis alur kerja

### 🏢 **Aplikasi Konkuren Perusahaan**
- **Pemrosesan Dokumen Volume Tinggi**: Memproses banyak dokumen secara simultan
- **Analisis Konten Waktu Nyata**: Analisis konkuren aliran data yang masuk
- **Optimasi Pemrosesan Batch**: Maksimalkan throughput untuk operasi skala besar
- **Analisis Multi-Modal**: Pemrosesan paralel atas berbagai tipe konten (teks, gambar, data)

## ⚙️ Prasyarat & Pengaturan

### 📦 **Ketergantungan yang Diperlukan**

Instal Agent Framework dengan kemampuan alur kerja konkuren:

```bash
pip install agent-framework -U
```

### 🔑 **Konfigurasi Microsoft Foundry**

Masuk dengan Azure CLI (`az login`) sehingga `AzureCliCredential` dapat mengautentikasi, kemudian atur detail proyek Microsoft Foundry Anda.

**Pengaturan Lingkungan (.env file):**
```env
AZURE_AI_PROJECT_ENDPOINT=https://<your-project>.services.ai.azure.com
AZURE_AI_MODEL_DEPLOYMENT_NAME=gpt-4.1-mini
```

**Pertimbangan Pemrosesan Konkuren:**
- **Batas Kecepatan**: Pantau batas kecepatan Azure OpenAI untuk permintaan konkuren
- **Penggunaan Sumber Daya**: Pertimbangkan penggunaan memori dan CPU dengan multi-agen konkuren
- **Penanganan Kesalahan**: Terapkan pemulihan kesalahan yang tangguh untuk operasi paralel

### 🏗️ **Arsitektur Alur Kerja Konkuren**

```mermaid
graph TD
    A[Mulai Alur Kerja] --> B[Eksekusi Bersamaan]
    B --> C[Kolam Agen 1]
    B --> D[Kolam Agen 2]
    B --> E[Kolam Agen 3]
    C --> F[Agregasi Hasil]
    D --> F
    E --> F
    F --> G[Output Akhir]
    
    H[Microsoft Foundry] --> C
    H --> D
    H --> E
```

**Manfaat Utama:**
- **⚡ Performa**: Percepatan signifikan melalui eksekusi paralel
- **📈 Skalabilitas**: Tangani peningkatan beban kerja tanpa peningkatan waktu yang proporsional
- **🔄 Efisiensi**: Pemanfaatan lebih baik dari sumber daya komputasi yang tersedia
- **🎯 Throughput**: Proses lebih banyak pekerjaan dalam waktu yang sama

## 🎨 **Pola Desain Alur Kerja Konkuren**

### 🔍 **Pipa Riset & Analisis**
```
Research Task → Parallel Research Agents → Content Synthesis → Quality Review
```

### 📊 **Alur Kerja Pemrosesan Data**
```
Input Data → Concurrent Processing Agents → Result Aggregation → Final Report
```

### 🎭 **Pipa Pembuatan Konten**
```
Content Brief → Parallel Content Generators → Review & Merge → Final Content
```

### 🔄 **Pemrosesan Multi-Tahap**
```
Input → Stage 1 (Concurrent) → Stage 2 (Concurrent) → Stage 3 (Sequential) → Output
```

## 🏢 **Manfaat Performa Perusahaan**

### ⚡ **Optimasi Throughput**
- **Eksekusi Paralel**: Beberapa agen bekerja secara simultan
- **Pemanfaatan Sumber Daya**: Efisiensi maksimum kapasitas model AI yang tersedia
- **Pengurangan Waktu**: Penurunan signifikan dalam total waktu pemrosesan
- **Arsitektur Skalabel**: Mudah menambah agen konkuren sesuai kebutuhan

### 🛡️ **Keandalan & Ketahanan**
- **Toleransi Kesalahan**: Kegagalan agen individual tidak menghentikan seluruh alur kerja
- **Isolasi Kesalahan**: Masalah pada satu cabang konkuren tidak mempengaruhi cabang lain
- **Penurunan Bertahap yang Elegan**: Sistem tetap beroperasi meski kapasitas agen berkurang
- **Mekanisme Pemulihan**: Percobaan ulang otomatis dan penanganan kesalahan untuk operasi gagal

### 📊 **Monitoring & Observabilitas**
- **Pelacakan Eksekusi Konkuren**: Pantau kemajuan semua operasi paralel
- **Metrik Performa**: Ukur percepatan dan peningkatan efisiensi
- **Analisis Penggunaan Sumber Daya**: Optimalkan alokasi agen konkuren
- **Identifikasi Hambatan**: Temukan dan selesaikan kendala performa

Mari bangun alur kerja AI konkuren berperforma tinggi! 🚀


In [ ]:
# Already covered by repo-level requirements.txt; left for reference.
# !pip install agent-framework -U

In [ ]:
import os
from typing import Any

from agent_framework import (
    Executor,
    Message,
    WorkflowBuilder,
    WorkflowContext,
    WorkflowViz,
    handler,
)
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from dotenv import load_dotenv

load_dotenv()


In [ ]:
# Configure the Microsoft Foundry client with keyless authentication.
# FoundryChatClient targets the Azure OpenAI Responses API.
provider = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=AzureCliCredential(),
)


In [ ]:
ResearcherAgentName = "Researcher-Agent"
ResearcherAgentInstructions = "You are my travel researcher, working with me to analyze the destination, list relevant attractions, and make detailed plans for each attraction."

In [ ]:
PlanAgentName = "Plan-Agent"
PlanAgentInstructions = "You are my travel planner, working with me to create a detailed travel plan based on the researcher's findings."

In [ ]:
research_agent = provider.as_agent(
    name=ResearcherAgentName,
    instructions=ResearcherAgentInstructions,
)

plan_agent = provider.as_agent(
    name=PlanAgentName,
    instructions=PlanAgentInstructions,
)


In [ ]:
# A passthrough executor that broadcasts the user input to every agent in parallel.
class InputDispatcher(Executor):
    """Forward the user input unchanged to all participating agents."""

    @handler
    async def forward(self, text: str, ctx: WorkflowContext[str]) -> None:
        await ctx.send_message(text)


dispatcher = InputDispatcher(id="dispatcher")
agents = [research_agent, plan_agent]

workflow = (
    WorkflowBuilder(
        start_executor=dispatcher,
        output_executors=agents,
    )
    .add_fan_out_edges(dispatcher, agents)
    .build()
)

In [ ]:
print("Generating workflow visualization...")
viz = WorkflowViz(workflow)
# Print out the mermaid string.
print("Mermaid string: \n=======")
print(viz.to_mermaid())
print("=======")
# Print out the DiGraph string.
print("DiGraph string: \n=======")
print(viz.to_digraph())
print("=======")
# SVG export needs the optional graphviz extra plus the graphviz system binary;
# fall back gracefully if it is not available.
try:
    svg_file = viz.export(format="svg")
    print(f"SVG file saved to: {svg_file}")
except ImportError as e:
    svg_file = None
    print(f"SVG export skipped (install graphviz to enable): {e}")

In [ ]:
from IPython.display import SVG, display, HTML
import os

print(f"Attempting to display SVG file at: {svg_file}")

if svg_file and os.path.exists(svg_file):
    try:
        # Preferred: direct SVG rendering
        display(SVG(filename=svg_file))
    except Exception as e:
        print(f"⚠️ Direct SVG render failed: {e}. Falling back to raw HTML.")
        try:
            with open(svg_file, "r", encoding="utf-8") as f:
                svg_text = f.read()
            display(HTML(svg_text))
        except Exception as inner:
            print(f"❌ Fallback HTML render also failed: {inner}")
else:
    print("❌ SVG file not found. Ensure viz.export(format='svg') ran successfully.")

In [ ]:
events = await workflow.run("Plan a trip to Seattle in December")
outputs = events.get_outputs()

In [ ]:
if outputs:
    print("===== Final Aggregated Responses =====")
    # outputs is a list of AgentResponse objects, one per output executor
    # (research_agent then plan_agent), in the order given to output_executors.
    for i, response in enumerate(outputs, start=1):
        print(f"{'-' * 60}\n\n{i:02d}:\n{response.text}")

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Penafian**:
Dokumen ini telah diterjemahkan menggunakan layanan terjemahan AI [Co-op Translator](https://github.com/Azure/co-op-translator). Meskipun kami berupaya untuk mencapai akurasi, harap diketahui bahwa terjemahan otomatis mungkin mengandung kesalahan atau ketidakakuratan. Dokumen asli dalam bahasa aslinya harus dianggap sebagai sumber yang sah. Untuk informasi penting, disarankan menggunakan terjemahan profesional oleh manusia. Kami tidak bertanggung jawab atas kesalahpahaman atau penafsiran yang keliru yang timbul dari penggunaan terjemahan ini.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
